# 03 · Preprocessing & Chunking

Clean the extracted Item 1A corpus and split it into model-ready **chunks** (≈ one risk
factor each). The chunking logic lives in `src/preprocess.py`; this notebook runs it and
performs quality-assurance checks.

Output: `data/processed/chunks_clean.csv`.

In [1]:
# Auto-reload edited src/ modules without restarting the kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import preprocess

## 1 · Load inputs and build chunks

In [2]:
corpus, sector_map = preprocess.load_inputs()
print(f"Corpus: {len(corpus)} filings, {corpus['cik'].nunique()} companies, "
      f"{corpus['year'].min()}-{corpus['year'].max()}")

chunks = preprocess.build_chunks(corpus, sector_map)

OUT = preprocess.OUTPUT_CSV
OUT.parent.mkdir(parents=True, exist_ok=True)
chunks.to_csv(OUT, index=False)
print(f"Built {len(chunks)} chunks -> {OUT.relative_to(preprocess.DATA_DIR.parent)}")

Corpus: 1661 filings, 109 companies, 2010-2025
Built 76581 chunks -> data/processed/chunks_clean.csv


## 2 · Overview

In [3]:
print(f"Chunks          : {len(chunks):,}")
print(f"Filings         : {len(corpus):,}")
print(f"Mean chunks/doc : {len(chunks)/len(corpus):.1f}")
print(f"Companies       : {chunks['cik'].nunique()}")
print(f"Sectors         : {chunks['sector'].nunique()}")
print("\nChunk word-count distribution:")
print(chunks['n_words'].describe().round(1).to_string())

Chunks          : 76,581
Filings         : 1,661
Mean chunks/doc : 46.1
Companies       : 109
Sectors         : 11

Chunk word-count distribution:
count    76581.0
mean       192.0
std        144.7
min         20.0
25%         56.0
50%        140.0
75%        370.0
max        858.0


## 3 · Quality assurance

These should all pass before topic modelling: no empty text, no out-of-range chunk
lengths, no missing metadata, and no chunk starting mid-sentence.

In [4]:
import re
problems = {}
problems['empty text']            = int((chunks['text'].str.strip() == '').sum())
problems['below MIN_CHUNK_WORDS'] = int((chunks['n_words'] < preprocess.MIN_CHUNK_WORDS).sum())
problems['above MAX_CHUNK_WORDS'] = int((chunks['n_words'] > preprocess.MAX_CHUNK_WORDS).sum())
problems['missing sector']        = int(chunks['sector'].isna().sum())
problems['starts lowercase/punct']= int((~chunks['text'].str.match(r'^[A-Z0-9"]')).sum())

for k, v in problems.items():
    flag = 'OK' if v == 0 else 'CHECK'
    print(f"  [{flag}] {k}: {v}")

  [OK] empty text: 0
  [OK] below MIN_CHUNK_WORDS: 0
  [CHECK] above MAX_CHUNK_WORDS: 140
  [OK] missing sector: 0
  [CHECK] starts lowercase/punct: 3910


## 4 · Chunks per year and per sector

Check temporal density (each year is a time-bin for BERTopic's topics-over-time) and that
no sector is missing. 2026 is expected to be thinner (partial filing cohort).

In [5]:
print('Chunks per year:')
print(chunks.groupby('year').size().to_string())
print('\nChunks per sector:')
print(chunks.groupby('sector').size().sort_values(ascending=False).to_string())

Chunks per year:
year
2010    8054
2011    7162
2012    7391
2013    5878
2014    4454
2015    4246
2016    3876
2017    4004
2018    3992
2019    3962
2020    3729
2021    3893
2022    3846
2023    3732
2024    4112
2025    4250

Chunks per sector:
sector
Real Estate               10007
Health Care                9257
Utilities                  8739
Financials                 7895
Information Technology     7617
Consumer Discretionary     7168
Communication Services     6900
Materials                  5854
Industrials                5004
Energy                     4606
Consumer Staples           3534


## 5 · Inspect sample chunks

In [6]:
for _, r in chunks.sample(3, random_state=0).iterrows():
    print(f"\n=== {r['ticker']} {r['year']} · chunk {r['chunk_id']} · {r['sector']} ({r['n_words']}w) ===")
    print('TEXT:', r['text'][:300])


=== WMB 2010 · chunk 35 · Energy (129w) ===
TEXT: Our Gas Pipeline and Midstream businesses provide some services pursuant to long-term, fixed price contracts. It is possible that costs to perform services under such contracts will exceed the revenues we collect for our services. Although most of the services provided by our interstate gas pipeline

=== PNW 2022 · chunk 9 · Utilities (393w) ===
TEXT: This is in large part due to a 2004 Arizona Court of Appeals decision that found critical components of the ACC’s rules to be violative of the Arizona Constitution. The ruling also voided the operating authority of all the competitive providers previously authorized by the ACC. On May 9, 2013, the A

=== COP 2022 · chunk 94 · Energy (36w) ===
TEXT: In addition, although we anticipate we will be able to repay our existing indebtedness when it matures or in accordance with our stated plans, there can be no assurance we will be able to do so.


---
Next: run BERTopic on filtered_corpus.csv (`src/model.py`).